Syntax

In [ ]:
#nn.Convo2d(in_channnels,output_channels,kernel_size,stride,padding)
'''
in_channels=means Number of layers in
ex:if black & white then in_channels=1
   if color is RGB then in_channels=3
written as [batch,channels,height,width] like [1,1,28,28]

output_channels
each output_channel = 1 pattern detector

kernel_size=how big the patch of a image of the filter jo 1 baari mai dekhe
if we pass 3 then 3x3 ke pixel mai dekhega
           5 then 5x5 ke pixel mai dekhega
smaller the kernel_size better the view so we use 3x3 

strides:means how much pixel we need to take jump to move next
ex: if stride=1 means jump 1 pixel at a time
ex: if stride=2 means jump 2 pixel at a time
higher the value make model faster

padding:means if we want to look at edge points or not
if we dont use padding then filter sahi terreke se image par nhi baith paayega and image is going to be shrink.it keep image stable
Ex:[1,1,28,28]<----passing i/p image
1->means whats the batch size or no. of images
1->no. of color layers(here black & white)
28->ht
28->width
conv=nn.Conv2d(
in_channel=1
out_channel=8
kernel_size=3
stride=1
padding=1
)
o/p is [1,8,28,28] for above parameters
o/p is [1,8,14,14] if stride=2(image got shrink)
o/p is[1,8,26,26] ff padding=0(the edges completely lost here)
'''

In [ ]:
import torch
import torch.nn as nn

image=torch.randn(1,1,28,28)
conv=nn.Conv2d(
    in_channels=1
    ,out_channels=8
    ,kernel_size=3
    ,stride=1
    ,padding=1
)
out=conv(image)
print("Input Imgae", image.shape)
print("Output Image", out.shape)

Pooling(in edit mode) 
In CNNs, pooling is a way to reduce the size of feature maps while keeping the important information.
Think of it as compressing an image's features.
Suppose a feature map is:

1  3  2  4
5  6  7  8
2  4  1  3
0  2  5  6
With 2×2 Max Pooling, we look at each 2×2 block and keep only the largest number.

First block:
1  3
5  6
Maximum = 6

Second:
2  4
7  8
Maximum = 8

Third:
2  4
0  2
Maximum = 4

Fourth:
1  3
5  6
Maximum = 6

So the result is:
6  8
4  6

We went from:
4 × 4
to:
2 × 2               while retaining strong features.

In [1]:
import torch
import torch.nn as nn

x=torch.randn(1,8,28,28)
pool=nn.MaxPool2d(kernel_size=2)
y=pool(x)
print("Before pooling", x.shape)
print("After pooling", y.shape)       # reduce shape since we reduce the image features

Before pooling torch.Size([1, 8, 28, 28])
After pooling torch.Size([1, 8, 14, 14])


Small project

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

transform = transforms.ToTensor()

train_data = MNIST(
    root="dat",
    train=True,
    download=True,
    transform=transform
)

test_data = MNIST(
    root="dat",
    train=False,
    download=True,
    transform=transform
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(16 * 7 * 7, 64),
            nn.ReLU(),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 3

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{num_epochs}] Loss: {total_loss / len(train_loader):.4f}")

# Evaluate on test set
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predicted = outputs.argmax(dim=1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")